<div style="display: flex; align-items: center;">

  <!-- Logos -->
  <div style="white-space: nowrap;">
    <img 
      src="https://www.upc.edu/comunicacio/ca/identitat/descarrega-arxius-grafics/fitxers-marca-principal/upc-positiu-p3005.png" 
      width="300"
      style="vertical-align: middle;"
    >
    <img 
      src="https://www.hipotecalowcost.com/wp-content/uploads/2019/08/Logo-CaixaBank.png" 
      width="200"
      style="vertical-align: middle;"
    >
  </div>

  <!-- Texto -->
  <div style="margin-left: auto; margin-right: 10px; text-align: right;">
      <p style="margin: 0;"><b>CaixaBank · Advanced Analytics Program</b></p>
      <p style="margin: 0;"><b>Model Risk & Data Science Training</b></p>
      <p style="margin: 0;">Automatización de procesos</p>
  </div>

</div>

# **Integración Python + n8n + APIs + Webhooks**
## *Conectar scripts Python con workflows n8n*

En este laboratorio construiremos el puente entre:

```text
Python
   ↕
HTTP
   ↕
n8n
   ↕
Webhook / API / otros servicios
```

La idea principal es que n8n orqueste eventos y servicios, mientras Python aporta lógica especializada cuando lo necesitamos.

Seguiremos:

> **preparar payload → validar → enviar → recibir respuesta → tratar errores → auditar → integrar en lote**

## Objetivos

Al finalizar deberías poder:

1. Crear payloads desde Python.
2. Enviar `POST` a un Webhook.
3. Utilizar `requests.post`.
4. Añadir headers.
5. Usar timeout.
6. Validar status codes.
7. Leer JSON de respuesta.
8. Capturar errores de conexión.
9. Implementar reintentos.
10. Enviar múltiples payloads.
11. Auditar llamadas.
12. Diseñar callbacks.
13. Entender integración bidireccional.
14. Preparar un microservicio Python para n8n.
15. Diseñar un workflow n8n que invoque Python vía HTTP.
16. Construir un mini-proyecto Python ↔ n8n.

## 1. Preparación

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys
import time
from datetime import datetime

import pandas as pd
import matplotlib.pyplot as plt

try:
    import requests
    REQUESTS_DISPONIBLE = True
except ImportError:
    REQUESTS_DISPONIBLE = False

RAIZ = Path(
    "demo_integracion_python_n8n"
)

if RAIZ.exists():
    shutil.rmtree(
        RAIZ
    )

for carpeta in [
    RAIZ / "payloads",
    RAIZ / "responses",
    RAIZ / "logs",
    RAIZ / "src"
]:
    carpeta.mkdir(
        parents=True,
        exist_ok=True
    )

print(
    "requests disponible:",
    REQUESTS_DISPONIBLE
)

print(
    "Proyecto:",
    RAIZ.resolve()
)

# 2. Arquitectura de integración

Existen dos direcciones principales.

## Python → n8n

```text
Python
  ↓ POST
Webhook n8n
  ↓
workflow
  ↓
respuesta
```

## n8n → Python

```text
workflow n8n
   ↓ HTTP Request
servicio Python
   ↓
respuesta JSON
```

En este laboratorio veremos ambas.

## 3. Crear payload desde Python

In [ ]:
payload = {
    "cliente_id":
        "C001",

    "cliente":
        "Empresa Norte",

    "tipo":
        "venta",

    "importe":
        2500.0,

    "email":
        "contacto@example.com",

    "timestamp":
        datetime.now().isoformat()
}

payload

## 3.1 Guardar payload

In [ ]:
ruta_payload = (
    RAIZ
    / "payloads"
    / "payload_demo.json"
)

ruta_payload.write_text(
    json.dumps(
        payload,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print(
    ruta_payload.read_text(
        encoding="utf-8"
    )
)

## 4. Validar payload antes de enviarlo

In [ ]:
CAMPOS_OBLIGATORIOS = {
    "cliente_id",
    "cliente",
    "tipo",
    "importe",
    "email"
}


def validar_payload(
    payload
):
    errores = []

    faltantes = (
        CAMPOS_OBLIGATORIOS
        - set(
            payload.keys()
        )
    )

    if faltantes:
        errores.append(
            f"Faltan campos: {sorted(faltantes)}"
        )

    if (
        "importe"
        in payload
        and payload[
            "importe"
        ] < 0
    ):
        errores.append(
            "importe no puede ser negativo"
        )

    if (
        "email"
        in payload
        and "@"
        not in str(
            payload[
                "email"
            ]
        )
    ):
        errores.append(
            "email no válido"
        )

    return errores


validar_payload(
    payload
)

## 4.1 Provocar payload inválido

In [ ]:
payload_invalido = {
    "cliente_id":
        "C002",
    "cliente":
        "Empresa Sur",
    "tipo":
        "venta",
    "importe":
        -200,
    "email":
        "correo_sin_arroba"
}

validar_payload(
    payload_invalido
)

### Interpretación

Antes de enviar datos a n8n conviene validar localmente lo que podamos.

Eso evita disparar workflows con información claramente inválida.

## 5. URL de Webhook

In [ ]:
WEBHOOK_URL = (
    "http://localhost:5678/webhook-test/"
    "clasificar-solicitud"
)

print(
    WEBHOOK_URL
)

En n8n, durante pruebas suele utilizarse una URL de **test webhook**.

En producción se utiliza la URL de producción del Webhook.

La URL real dependerá de tu instancia.

## 6. Función Python → Webhook

In [ ]:
def enviar_webhook(
    url,
    payload,
    timeout=10
):
    if not REQUESTS_DISPONIBLE:
        return {
            "ok":
                False,
            "status":
                None,
            "json":
                None,
            "error":
                "requests no está instalado"
        }

    errores = validar_payload(
        payload
    )

    if errores:
        return {
            "ok":
                False,
            "status":
                None,
            "json":
                None,
            "error":
                "; ".join(
                    errores
                )
        }

    try:
        respuesta = requests.post(
            url,
            json=payload,
            headers={
                "Content-Type":
                    "application/json",
                "User-Agent":
                    "DataGathery-Python/1.0"
            },
            timeout=timeout
        )

        respuesta.raise_for_status()

        try:
            contenido = (
                respuesta.json()
            )
        except ValueError:
            contenido = {
                "text":
                    respuesta.text
            }

        return {
            "ok":
                True,
            "status":
                respuesta.status_code,
            "json":
                contenido,
            "error":
                None
        }

    except requests.Timeout:
        return {
            "ok":
                False,
            "status":
                None,
            "json":
                None,
            "error":
                "Timeout"
        }

    except requests.ConnectionError as error:
        return {
            "ok":
                False,
            "status":
                None,
            "json":
                None,
            "error":
                f"ConnectionError: {error}"
        }

    except requests.HTTPError as error:
        return {
            "ok":
                False,
            "status":
                error.response.status_code
                if error.response
                is not None
                else None,
            "json":
                None,
            "error":
                f"HTTPError: {error}"
        }

    except requests.RequestException as error:
        return {
            "ok":
                False,
            "status":
                None,
            "json":
                None,
            "error":
                f"RequestException: {error}"
        }

## 6.1 Ejecutar llamada

In [ ]:
resultado_webhook = enviar_webhook(
    WEBHOOK_URL,
    payload,
    timeout=3
)

resultado_webhook

### Interpretación

Si n8n no está activo, veremos un `ConnectionError`.

Eso también es útil pedagógicamente: primero observamos el fallo real y después activamos el workflow.

## 7. Simular respuesta n8n

In [ ]:
respuesta_simulada_n8n = {
    "status":
        "ok",

    "cliente_id":
        "C001",

    "clasificacion":
        "alta_prioridad",

    "accion":
        "revisar"
}

respuesta_simulada_n8n

## 7.1 Validar respuesta

In [ ]:
CAMPOS_RESPUESTA = {
    "status",
    "cliente_id",
    "clasificacion",
    "accion"
}


def validar_respuesta_n8n(
    respuesta
):
    if not isinstance(
        respuesta,
        dict
    ):
        return [
            "La respuesta no es un objeto JSON."
        ]

    faltantes = (
        CAMPOS_RESPUESTA
        - set(
            respuesta.keys()
        )
    )

    if faltantes:
        return [
            f"Faltan campos: {sorted(faltantes)}"
        ]

    return []


validar_respuesta_n8n(
    respuesta_simulada_n8n
)

## 8. Diseñar el workflow correspondiente en n8n

Workflow sugerido:

```text
Webhook
   ↓
Edit Fields
   ↓
IF importe >= 1000
 ┌───────────────┐
 ↓               ↓
alta_prioridad   normal
 └──────┬────────┘
        ↓
Respond to Webhook
```

La respuesta podría ser:

```json
{
  "status": "ok",
  "cliente_id": "{{$json.cliente_id}}",
  "clasificacion": "alta_prioridad",
  "accion": "revisar"
}
```

## 9. Envío de lote

In [ ]:
solicitudes = pd.DataFrame({
    "cliente_id": [
        "C001",
        "C002",
        "C003",
        "C004",
        "C005"
    ],

    "cliente": [
        "Empresa A",
        "Empresa B",
        "Empresa C",
        "Empresa D",
        "Empresa E"
    ],

    "tipo": [
        "venta",
        "soporte",
        "venta",
        "otro",
        "soporte"
    ],

    "importe": [
        2500,
        200,
        800,
        5000,
        1200
    ],

    "email": [
        "a@example.com",
        "b@example.com",
        "c@example.com",
        "d@example.com",
        "e@example.com"
    ]
})

solicitudes

## 9.1 Convertir cada fila a payload

In [ ]:
payloads = []

for _, fila in solicitudes.iterrows():

    payload_individual = (
        fila.to_dict()
    )

    payload_individual[
        "timestamp"
    ] = datetime.now().isoformat()

    payloads.append(
        payload_individual
    )

payloads[:2]

## 9.2 Guardar lote de payloads

In [ ]:
for payload_individual in payloads:

    ruta = (
        RAIZ
        / "payloads"
        / f'{payload_individual["cliente_id"]}.json'
    )

    ruta.write_text(
        json.dumps(
            payload_individual,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

print(
    "Payloads guardados:",
    len(
        list(
            (
                RAIZ
                / "payloads"
            ).glob(
                "C*.json"
            )
        )
    )
)

## 10. Auditoría de llamadas

In [ ]:
def enviar_lote(
    url,
    payloads,
    timeout=5
):
    auditoria = []

    for payload in payloads:

        inicio = (
            time.perf_counter()
        )

        resultado = (
            enviar_webhook(
                url,
                payload,
                timeout=timeout
            )
        )

        auditoria.append({
            "cliente_id":
                payload[
                    "cliente_id"
                ],

            "ok":
                resultado[
                    "ok"
                ],

            "status":
                resultado[
                    "status"
                ],

            "error":
                resultado[
                    "error"
                ],

            "segundos":
                (
                    time.perf_counter()
                    - inicio
                )
        })

    return pd.DataFrame(
        auditoria
    )

## 10.1 Ejecutar lote

In [ ]:
auditoria_lote = enviar_lote(
    WEBHOOK_URL,
    payloads,
    timeout=2
)

auditoria_lote

## 10.2 Visualizar éxito/error

In [ ]:
conteo_estado = pd.Series({
    "ok":
        int(
            auditoria_lote[
                "ok"
            ].sum()
        ),

    "error":
        int(
            (
                ~auditoria_lote[
                    "ok"
                ]
            ).sum()
        )
})

plt.figure(
    figsize=(7, 5)
)

plt.bar(
    conteo_estado.index,
    conteo_estado.values
)

plt.ylabel(
    "Llamadas"
)

plt.title(
    "Resultado del envío a n8n"
)

plt.tight_layout()
plt.show()

## 11. Reintentos

In [ ]:
def enviar_webhook_con_reintentos(
    url,
    payload,
    intentos=3,
    espera=1,
    timeout=5
):
    ultimo = None

    for intento in range(
        1,
        intentos + 1
    ):

        resultado = enviar_webhook(
            url,
            payload,
            timeout=timeout
        )

        ultimo = resultado

        if resultado[
            "ok"
        ]:
            resultado[
                "intentos"
            ] = intento

            return resultado

        print(
            f"Intento {intento} fallido:",
            resultado[
                "error"
            ]
        )

        if intento < intentos:
            time.sleep(
                espera
            )

    ultimo[
        "intentos"
    ] = intentos

    return ultimo

## 11.1 Ejecutar reintento

In [ ]:
resultado_reintento = (
    enviar_webhook_con_reintentos(
        WEBHOOK_URL,
        payload,
        intentos=2,
        espera=0.5,
        timeout=2
    )
)

resultado_reintento

### Interpretación

No todos los errores son reintentables.

Tiene sentido reintentar:

- timeout;
- conexión temporal;
- 429;
- algunos 5xx.

Normalmente no tiene sentido reintentar:

- 400;
- payload inválido;
- credenciales incorrectas.

## 12. Clasificar errores reintentables

In [ ]:
def es_reintentable(
    resultado
):
    if resultado[
        "ok"
    ]:
        return False

    status = (
        resultado[
            "status"
        ]
    )

    error = (
        resultado[
            "error"
        ]
        or ""
    )

    if (
        "Timeout"
        in error
        or "ConnectionError"
        in error
    ):
        return True

    if status in {
        429,
        500,
        502,
        503,
        504
    }:
        return True

    return False


es_reintentable(
    resultado_webhook
)

# 13. Callback: n8n → Python

Un workflow puede también llamar a un servicio Python.

Ejemplo:

```text
Webhook n8n
    ↓
datos
    ↓
HTTP Request
    ↓
API Python
    ↓
resultado
    ↓
n8n
```

Para ello necesitamos exponer Python vía HTTP.

## 14. Microservicio Python conceptual con FastAPI

In [ ]:
codigo_fastapi = '''
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class Solicitud(BaseModel):
    cliente_id: str
    importe: float

@app.post("/clasificar")
def clasificar(solicitud: Solicitud):
    prioridad = (
        "alta"
        if solicitud.importe >= 1000
        else "normal"
    )

    return {
        "cliente_id":
            solicitud.cliente_id,
        "prioridad":
            prioridad
    }
'''

ruta_fastapi = (
    RAIZ
    / "src"
    / "api_python.py"
)

ruta_fastapi.write_text(
    textwrap.dedent(
        codigo_fastapi
    ),
    encoding="utf-8"
)

print(
    ruta_fastapi.read_text(
        encoding="utf-8"
    )
)

## 14.1 Ejecutar microservicio

Con FastAPI y Uvicorn instalados:

```bash
uvicorn api_python:app --reload --app-dir src
```

Después n8n puede usar HTTP Request:

```text
POST http://host.docker.internal:8000/clasificar
```

si n8n corre en Docker sobre Windows/macOS.

La dirección exacta depende de cómo estén desplegados ambos servicios.

## 15. Payload n8n → Python

In [ ]:
payload_python = {
    "cliente_id":
        "C001",

    "importe":
        2500
}

print(
    json.dumps(
        payload_python,
        indent=2
    )
)

## 16. Respuesta esperada Python

In [ ]:
respuesta_python = {
    "cliente_id":
        "C001",

    "prioridad":
        "alta"
}

respuesta_python

## 17. Diseñar integración bidireccional

Arquitectura:

```text
Python script
    ↓
Webhook n8n
    ↓
validar
    ↓
HTTP Request
    ↓
FastAPI Python
    ↓
clasificación
    ↓
n8n
    ↓
Respond to Webhook
    ↓
Python script
```

Esto permite:

- orquestación visual;
- lógica Python especializada;
- respuesta síncrona.

## 18. Detectar herramientas locales

In [ ]:
herramientas = {
    "python":
        sys.executable,

    "docker":
        shutil.which(
            "docker"
        ),

    "n8n":
        shutil.which(
            "n8n"
        ),

    "uvicorn":
        shutil.which(
            "uvicorn"
        )
}

herramientas

## 19. Construir comando curl

In [ ]:
def construir_curl(
    url,
    payload
):
    contenido = json.dumps(
        payload,
        ensure_ascii=False
    )

    return (
        f'curl -X POST "{url}" '
        f'-H "Content-Type: application/json" '
        f"-d '{contenido}'"
    )


print(
    construir_curl(
        WEBHOOK_URL,
        payload
    )
)

## 20. Preparar JSON para importar/exportar pruebas

In [ ]:
especificacion = {
    "workflow":
        "Python n8n integration",

    "entrada":
        {
            "metodo":
                "POST",

            "campos": [
                "cliente_id",
                "cliente",
                "tipo",
                "importe",
                "email"
            ]
        },

    "nodos": [
        "Webhook",
        "Edit Fields",
        "IF",
        "HTTP Request Python",
        "Respond to Webhook"
    ],

    "salida": [
        "status",
        "cliente_id",
        "clasificacion",
        "accion"
    ]
}

ruta_spec = (
    RAIZ
    / "workflow_spec.json"
)

ruta_spec.write_text(
    json.dumps(
        especificacion,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print(
    ruta_spec.read_text(
        encoding="utf-8"
    )
)

## 21. Persistir respuestas

In [ ]:
def guardar_respuesta(
    cliente_id,
    respuesta
):
    ruta = (
        RAIZ
        / "responses"
        / f"{cliente_id}.json"
    )

    ruta.write_text(
        json.dumps(
            respuesta,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

    return ruta


ruta_respuesta = guardar_respuesta(
    "C001",
    respuesta_simulada_n8n
)

print(
    ruta_respuesta
)

## 22. Auditoría completa

In [ ]:
registro_auditoria = {
    "timestamp":
        datetime.now().isoformat(),

    "cliente_id":
        payload[
            "cliente_id"
        ],

    "webhook":
        WEBHOOK_URL,

    "resultado_ok":
        resultado_webhook[
            "ok"
        ],

    "status":
        resultado_webhook[
            "status"
        ],

    "error":
        resultado_webhook[
            "error"
        ]
}

registro_auditoria

## 22.1 Guardar auditoría CSV

In [ ]:
ruta_log = (
    RAIZ
    / "logs"
    / "auditoria.csv"
)

pd.DataFrame(
    [
        registro_auditoria
    ]
).to_csv(
    ruta_log,
    index=False
)

print(
    ruta_log
)

## 23. Mini-proyecto integrado

Construye:

```text
Python
   ↓ POST
Webhook n8n
   ↓
validar
   ↓
IF importe >= 1000
   ↓
HTTP Request a FastAPI Python
   ↓
clasificación adicional
   ↓
Respond to Webhook
   ↓
Python guarda respuesta
```

Debe incluir:

1. payload validado;
2. webhook;
3. timeout;
4. gestión de errores;
5. reintentos;
6. respuesta JSON;
7. microservicio Python;
8. HTTP Request desde n8n;
9. auditoría;
10. prueba con al menos 10 solicitudes.

## 24. Checklist de integración

- [ ] ¿El payload tiene esquema definido?
- [ ] ¿Se valida antes de enviar?
- [ ] ¿Uso POST cuando corresponde?
- [ ] ¿Uso JSON?
- [ ] ¿Existe timeout?
- [ ] ¿Compruebo status code?
- [ ] ¿Valido la respuesta?
- [ ] ¿Diferencio errores reintentables?
- [ ] ¿Registro cada llamada?
- [ ] ¿El webhook devuelve una respuesta clara?
- [ ] ¿n8n puede llamar a Python?
- [ ] ¿Python puede llamar a n8n?
- [ ] ¿Las URLs cambian entre test y producción?
- [ ] ¿Las credenciales están protegidas?

## 25. Ejercicios finales

### Ejercicio 1
Crea un payload.

### Ejercicio 2
Valida campos obligatorios.

### Ejercicio 3
Envía POST a un webhook.

### Ejercicio 4
Añade headers.

### Ejercicio 5
Añade timeout.

### Ejercicio 6
Captura ConnectionError.

### Ejercicio 7
Captura HTTPError.

### Ejercicio 8
Valida JSON de respuesta.

### Ejercicio 9
Genera 10 payloads.

### Ejercicio 10
Envía lote.

### Ejercicio 11
Audita tiempos.

### Ejercicio 12
Implementa reintentos.

### Ejercicio 13
Clasifica errores reintentables.

### Ejercicio 14
Crea FastAPI mínima.

### Ejercicio 15
Haz HTTP Request desde n8n a Python.

### Ejercicio 16
Construye integración bidireccional.

### Ejercicio 17
Guarda respuestas.

### Ejercicio 18
Reto: construye el mini-proyecto Python ↔ n8n completo.

## 26. Preguntas de reflexión

1. ¿Qué aporta un Webhook?
2. ¿Qué diferencia hay entre Python → n8n y n8n → Python?
3. ¿Por qué validar payload?
4. ¿Por qué usar timeout?
5. ¿Cuándo reintentar?
6. ¿Qué status codes no reintentarías?
7. ¿Qué aporta FastAPI?
8. ¿Qué papel cumple n8n?
9. ¿Qué papel cumple Python?
10. ¿Qué debe registrarse?
11. ¿Qué diferencia existe entre webhook test y producción?
12. ¿Cómo desplegarías ambos componentes?
13. ¿Dónde guardarías secretos?
14. ¿Cómo convertirías este flujo en una automatización real de negocio?

## 27. Cierre

Ya hemos conectado:

```text
Python
  ↕
HTTP
  ↕
n8n
```

El siguiente paso será añadir inteligencia:

```text
workflow
   ↓
LLM
   ↓
prompt
   ↓
decisión
   ↓
agente
```

y empezaremos con **LLM, prompting y agentes**.